# 02 · Revisión de todas las fuentes

**Objetivo:** documentar qué contiene cada fuente: rango de fechas, bandas, resolución, rol en el proyecto y limitaciones.

**Solo lee datos.** No escribe archivos. Las consultas a GEE son de metadatos.

**Estado:** notebook nuevo, todavía sin ejecutar.

**Orden del proyecto:** 01 límite → **02 fuentes** → 03 viabilidad de años → 04 catálogo de indicadores → 05 visor de imágenes → 06 descarga.


## 1. Configuración


In [14]:
from datetime import date, datetime
from pathlib import Path
import hashlib
import io
import json
import re

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

GEE_PROJECT = "viirs-peru"
ANIO_INI, ANIO_FIN = 2010, 2025  # 15 periodos anuales: 2010 -> 2025

# Bandas por sensor: nombre común -> banda original
BANDAS_LANDSAT = {
    "L5": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L7": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L8": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
    "L9": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
}
BANDAS_S2 = {"blue": "B2", "green": "B3", "red": "B4", "nir": "B8", "swir1": "B11", "swir2": "B12"}
COLECCIONES_LANDSAT = {
    "L5": "LANDSAT/LT05/C02/T1_L2",
    "L7": "LANDSAT/LE07/C02/T1_L2",
    "L8": "LANDSAT/LC08/C02/T1_L2",
    "L9": "LANDSAT/LC09/C02/T1_L2",
}
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial).
    Se usa el GeoPackage como marcador porque puede existir una carpeta data/raw dentro de notebooks/."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def rango_fechas(col):
    """Primera y última fecha de una colección, como texto."""
    d = ee.Dictionary({
        "min": col.aggregate_min("system:time_start"),
        "max": col.aggregate_max("system:time_start"),
    }).getInfo()
    if d["min"] is None:
        return "sin fecha"
    a = pd.to_datetime(d["min"], unit="ms").strftime("%Y-%m-%d")
    b = pd.to_datetime(d["max"], unit="ms").strftime("%Y-%m-%d")
    return f"{a} → {b}"


def md5(ruta):
    """Suma MD5 de un archivo, leído por bloques."""
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


iniciar_gee()
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"
gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
print("Provincia:", prov["nombprov"].iloc[0], "| ventana:", ANIO_INI, "-", ANIO_FIN)


Provincia: CUSCO | ventana: 2010 - 2025


## 2. Límite provincial (INEI)

El límite viene de **INEI, límites provinciales** (`data/raw/Provincia/PROVINCIA.gpkg`). La columna `fuente` del archivo dice "V Censo Nacional Económico"; ese dato se documenta tal cual.


In [15]:
FILAS = []  # cada sección agrega una fila a la tabla final

print("CRS:", gdf.crs)
print("Columna fuente:", prov["fuente"].iloc[0])
print("Área (km², EPSG:32719):", round(prov.to_crs(epsg=32719).area.iloc[0] / 1e6, 1))
print("Límites (lon/lat):", [round(v, 4) for v in prov.total_bounds])


CRS: EPSG:4326
Columna fuente: V Censo Nacional Economico
Área (km², EPSG:32719): 530.1
Límites (lon/lat): [np.float64(-72.168), np.float64(-13.6492), np.float64(-71.8046), np.float64(-13.4295)]


## 3. Sentinel-2 SR Harmonized


In [16]:
col = ee.ImageCollection(COLECCION_S2).filterBounds(aoi)
img = ee.Image(col.first())
FILAS.append({
    "fuente": "Sentinel-2 SR Harmonized",
    "producto": "reflectancia superficial",
    "rango_fechas": rango_fechas(col),
    "escala_m": img.select(0).projection().nominalScale().getInfo(),
    "bandas": ", ".join(img.bandNames().getInfo()),
    "rol": "complemento (2019–2025)",
})
print("Bandas que usaremos (nombre común -> banda):", BANDAS_S2)


Bandas que usaremos (nombre común -> banda): {'blue': 'B2', 'green': 'B3', 'red': 'B4', 'nir': 'B8', 'swir1': 'B11', 'swir2': 'B12'}


## 4. Landsat C2 L2 por sensor

Los cuatro sensores se estandarizan con la misma escala y los mismos nombres de banda. Cada sensor tiene números de banda distintos.


In [17]:
for sensor, cid in COLECCIONES_LANDSAT.items():
    col = ee.ImageCollection(cid).filterBounds(aoi)
    img = ee.Image(col.first())
    FILAS.append({
        "fuente": f"Landsat {sensor[1]} C2 L2",
        "producto": "reflectancia superficial",
        "rango_fechas": rango_fechas(col),
        "escala_m": img.select(0).projection().nominalScale().getInfo(),
        "bandas": ", ".join(img.bandNames().getInfo()),
        "rol": "serie principal",
    })

print("Bandas por sensor (nombre común -> banda original):")
print(pd.DataFrame(BANDAS_LANDSAT).T)


Bandas por sensor (nombre común -> banda original):
     blue  green    red    nir  swir1  swir2
L5  SR_B1  SR_B2  SR_B3  SR_B4  SR_B5  SR_B7
L7  SR_B1  SR_B2  SR_B3  SR_B4  SR_B5  SR_B7
L8  SR_B2  SR_B3  SR_B4  SR_B5  SR_B6  SR_B7
L9  SR_B2  SR_B3  SR_B4  SR_B5  SR_B6  SR_B7


## 5. ESA WorldCover (2020 y 2021)


In [18]:
for etiqueta, cid in [("WorldCover v100 (2020)", "ESA/WorldCover/v100"),
                      ("WorldCover v200 (2021)", "ESA/WorldCover/v200")]:
    col = ee.ImageCollection(cid).filterBounds(aoi)
    img = ee.Image(col.first())
    FILAS.append({
        "fuente": etiqueta,
        "producto": "clases de cobertura",
        "rango_fechas": rango_fechas(col),
        "escala_m": img.select(0).projection().nominalScale().getInfo(),
        "bandas": ", ".join(img.bandNames().getInfo()),
        "rol": "referencia puntual",
    })


## 6. SRTM (elevación)


In [19]:
img = ee.Image("USGS/SRTMGL1_003")
FILAS.append({
    "fuente": "SRTM",
    "producto": "modelo de elevación",
    "rango_fechas": "sin fecha (modelo fijo)",
    "escala_m": img.select(0).projection().nominalScale().getInfo(),
    "bandas": ", ".join(img.bandNames().getInfo()),
    "rol": "factor (altitud)",
})


## 7. MapBiomas Perú Colección 4


In [20]:
mb = ee.Image(MB_ASSET)
bandas_mb = mb.bandNames().getInfo()
anios_mb = sorted(int(b.split("_")[1]) for b in bandas_mb if b.startswith("classification_"))
FILAS.append({
    "fuente": "MapBiomas Perú C4",
    "producto": "cobertura y uso del suelo",
    "rango_fechas": f"{anios_mb[0]} a {anios_mb[-1]} (una banda por año)",
    "escala_m": mb.select(0).projection().nominalScale().getInfo(),
    "bandas": f"{len(bandas_mb)} bandas classification_YYYY",
    "rol": "referencia principal",
})


## 8. GEOBOSQUES (archivos locales)

Se revisa lo que ya tenemos en `data/raw/GEOBOSQUES/`: la estructura del JSON, los años de las imágenes y si hay imágenes repetidas.


In [21]:
def describir(obj, prefijo="", nivel=0, max_nivel=3):
    """Imprime la estructura de un JSON: claves, tipos y un valor corto."""
    if nivel > max_nivel:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            vista = "" if isinstance(v, (dict, list)) else f" = {str(v)[:80]}"
            print(f"{prefijo}{k} ({type(v).__name__}){vista}")
            describir(v, prefijo + "  ", nivel + 1, max_nivel)
    elif isinstance(obj, list):
        print(f"{prefijo}[lista de {len(obj)}]")
        if obj:
            describir(obj[0], prefijo + "  ", nivel + 1, max_nivel)


carpeta_gb = RAIZ / "data" / "raw" / "GEOBOSQUES"
ruta_json = carpeta_gb / "info_bosque_humedo_2025.json"
if ruta_json.exists():
    info_gb = json.loads(ruta_json.read_text(encoding="utf-8"))
    print("Estructura de info_bosque_humedo_2025.json:")
    describir(info_gb)
else:
    print("No encuentro", ruta_json)


Estructura de info_bosque_humedo_2025.json:
currentVersion (float) = 11.5
cimVersion (str) = 3.5.0
serviceDescription (str) = 
mapName (str) = serv_geobosques_interop_bosque_humedo_2025
description (str) = 
copyrightText (str) = 
supportsDynamicLayers (bool) = True
layers (list)
  [lista de 6]
    id (int) = 1
    name (str) = Pérdida antrópica (Deforestación) 2025
    parentLayerId (int) = -1
    defaultVisibility (bool) = True
    subLayerIds (NoneType) = None
    minScale (int) = 0
    maxScale (int) = 0
    type (str) = Raster Layer
    supportsDynamicLegends (bool) = True
tables (list)
  [lista de 0]
spatialReference (dict)
  wkid (int) = 102100
  latestWkid (int) = 3857
  xyTolerance (float) = 0.001
  zTolerance (float) = 0.001
  mTolerance (float) = 0.001
  falseX (int) = -20037700
  falseY (int) = -30241100
  xyUnits (int) = 10000
  falseZ (int) = -100000
  zUnits (int) = 10000
  falseM (int) = -100000
  mUnits (int) = 10000
singleFusedMapCache (bool) = True
tileInfo (dict)
  r

In [22]:
pngs = sorted((carpeta_gb / "anual").glob("bosque_*_provincia_cusco.png"))
anios_png = sorted(int(re.search(r"bosque_(\d{4})_", p.name).group(1)) for p in pngs)
faltan = [a for a in range(2016, 2026) if a not in anios_png]
print("Años con imagen:", anios_png)
print("Años faltantes entre 2016 y 2025:", faltan or "ninguno")

verificacion = pd.DataFrame({
    "archivo": [p.name for p in pngs],
    "bytes": [p.stat().st_size for p in pngs],
    "md5": [md5(p) for p in pngs],
})
print(verificacion)
print("Imágenes con MD5 repetido:", int(verificacion["md5"].duplicated(keep=False).sum()))

FILAS.append({
    "fuente": "GEOBOSQUES (imágenes WMS)",
    "producto": "bosque y pérdida de bosque (visual)",
    "rango_fechas": f"imágenes locales: {anios_png[0]} a {anios_png[-1]}" if anios_png else "sin imágenes",
    "escala_m": "no aplica (PNG)",
    "bandas": "no aplica",
    "rol": "referencia (cobertura parcial)",
})


Años con imagen: [2016, 2017, 2018, 2019, 2020, 2022, 2023, 2024, 2025]
Años faltantes entre 2016 y 2025: [2021]
                           archivo  bytes                               md5
0  bosque_2016_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
1  bosque_2017_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
2  bosque_2018_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
3  bosque_2019_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
4  bosque_2020_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
5  bosque_2022_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
6  bosque_2023_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
7  bosque_2024_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
8  bosque_2025_provincia_cusco.png  10301  ac33384d5a7810e958d0db6d47875efa
Imágenes con MD5 repetido: 9


In [23]:
import xml.etree.ElementTree as ET

WMS_URL = "https://gis.bosques.gob.pe/server/services/Interoperabilidad/bosque_humedo_2025/MapServer/WMSServer"
resp = requests.get(WMS_URL, params={"service": "WMS", "request": "GetCapabilities"}, timeout=120)
resp.raise_for_status()
raiz_xml = ET.fromstring(resp.content)


def etiqueta_xml(el):
    return el.tag.split("}")[-1]


capas_wms = []
for el in raiz_xml.iter():
    if etiqueta_xml(el) == "Layer":
        nombre = next((c.text for c in el if etiqueta_xml(c) == "Name"), None)
        titulo = next((c.text for c in el if etiqueta_xml(c) == "Title"), None)
        if nombre:
            capas_wms.append({"Name": nombre, "Title": titulo})

capas_wms = pd.DataFrame(capas_wms)
print("Capas del WMS:")
print(capas_wms)

# La capa se elige por su título. En la tabla de arriba es el Name "2" ("Bosque al 2025").
TITULO_CAPA = "Bosque al 2025"
coincide = capas_wms[capas_wms["Title"] == TITULO_CAPA]
CAPA_WMS = coincide["Name"].iloc[0] if len(coincide) == 1 else None
print("CAPA_WMS:", CAPA_WMS)


Capas del WMS:
  Name                                   Title
0    0                       No bosque al 2000
1    1                             Hidrografía
2    2                          Bosque al 2025
3    3                     Pérdida 2001 - 2024
4    4                    Pérdida natural 2025
5    5  Pérdida antrópica (Deforestación) 2025
CAPA_WMS: 2


In [24]:
if CAPA_WMS is not None:
    import geemap
    from ipyleaflet import WMSLayer

    mapa_gb = geemap.Map()
    mapa_gb.centerObject(aoi, 9)
    mapa_gb.add_layer(WMSLayer(url=WMS_URL, layers=CAPA_WMS, format="image/png",
                               transparent=True, name="GeoBosques 2025"))
    mapa_gb.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2),
                     {"palette": "000000"}, "Límite provincia")
    mapa_gb
else:
    print(f"No encontré la capa '{TITULO_CAPA}' en el WMS. Revisa la tabla de la celda anterior.")


## 9. OpenStreetMap (red vial)

Es una foto del 2026-10-08. No tiene historial por año.


In [25]:
import pyogrio

OSM_GPKG = RAIZ / "data" / "raw" / "OpenStreetMap" / "peru.gpkg"
readme = RAIZ / "data" / "raw" / "OpenStreetMap" / "README"
if readme.exists():
    print(readme.read_text(encoding="utf-8", errors="replace")[:600])

capas_osm = [c[0] for c in pyogrio.list_layers(OSM_GPKG)]
print("Capas en peru.gpkg:", len(capas_osm))

info_vias = pyogrio.read_info(OSM_GPKG, layer="gis_osm_roads_free")
print("Tramos de vía (registros):", info_vias["features"])
print("Campos:", list(info_vias["fields"]))

FILAS.append({
    "fuente": "OpenStreetMap (red vial)",
    "producto": "red vial",
    "rango_fechas": "foto 2026-10-08, sin historial",
    "escala_m": "vectorial",
    "bandas": "no aplica",
    "rol": "factor (distancia a vías), sin historial",
})


The GPKG file in this archive has been created from OpenStreetMap data
and is licensed under the Open Database 1.0 License. See 
www.openstreetmap.org for details about the project.

This file contains OpenStreetMap data as of 2026-10-08T20:21:06Z
. Every day
a new version of this file will be made available at:

https://download.geofabrik.de/south-america/peru-latest-free.gpkg.zip

A documentation of the layers in this file is available here:

https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf

Geofabrik also makes extended data exports to order; please see
https://www.geofabrik.d
Capas en peru.gpkg: 20
Tramos de vía (registros): 914586
Campos: ['osm_id', 'code', 'fclass', 'name', 'ref', 'oneway', 'maxspeed', 'layer', 'bridge', 'tunnel']


## 10. Tabla resumen de fuentes

Revisa la columna `rol` y las limitaciones antes de pasar al notebook 03.


In [26]:
revision = pd.DataFrame(FILAS)
revision


,fuente,producto,rango_fechas,escala_m,bandas,rol
0,Sentinel-2 SR Harmonized,reflectancia superficial,2016-04-30 → 2026-10-07,60,"B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, ...",complemento (2019–2025)
1,Landsat 5 C2 L2,reflectancia superficial,1984-08-30 → 2011-10-28,30,"SR_B1, SR_B2, SR_B3, SR_B4, SR_B5, SR_B7, SR_A...",serie principal
2,Landsat 7 C2 L2,reflectancia superficial,1999-08-16 → 2023-12-09,30,"SR_B1, SR_B2, SR_B3, SR_B4, SR_B5, SR_B7, SR_A...",serie principal
3,Landsat 8 C2 L2,reflectancia superficial,2013-03-26 → 2026-09-19,30,"SR_B1, SR_B2, SR_B3, SR_B4, SR_B5, SR_B6, SR_B...",serie principal
4,Landsat 9 C2 L2,reflectancia superficial,2021-11-05 → 2026-09-27,30,"SR_B1, SR_B2, SR_B3, SR_B4, SR_B5, SR_B6, SR_B...",serie principal
5,WorldCover v100 (2020),clases de cobertura,2020-01-01 → 2020-01-01,9.276624,Map,referencia puntual
6,WorldCover v200 (2021),clases de cobertura,2021-01-01 → 2021-01-01,9.276624,Map,referencia puntual
7,SRTM,modelo de elevación,sin fecha (modelo fijo),30.922081,elevation,factor (altitud)
8,MapBiomas Perú C4,cobertura y uso del suelo,1985 a 2025 (una banda por año),30.0,41 bandas classification_YYYY,referencia principal
9,GEOBOSQUES (imágenes WMS),bosque y pérdida de bosque (visual),imágenes locales: 2016 a 2025,no aplica (PNG),no aplica,referencia (cobertura parcial)
